# Devign: no_c8_mmr_demo_selection — code-based v2

Dùng cache CodeT5/Joern đã xác minh của Reproduced-Code-Based. Chọn kernel WSL, Restart Kernel rồi Run All. Output mới: No-C8-mmr-demo-selection/test_v2/. Cache cũ trong test/ được giữ. Reveal cần baseline prepare/freeze hoàn tất trước.

C6 dùng PDG có sẵn, có thay đổi nguồn biểu diễn graph so với cache legacy; manifest ghi rõ. C8 chỉ có tác dụng đa dạng hóa khi C2 bật nhiều demo. Xem README.md trong thư mục Ablations.


In [3]:
from pathlib import Path
import os, sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path('/mnt/c/MY-FILES/LAB-THAY-THO'), Path('C:/MY-FILES/LAB-THAY-THO')) if (p / 'Graduation-Project/Ablation_V2').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot locate workspace root')
ABLATIONS = ROOT / 'Graduation-Project/Ablation_V2/Devign/Ablations'
sys.path.insert(0, str(ABLATIONS))
from code_based_runner import AblationConfig, ARM_SPECS, run_stage
import code_based_runner as _lock_runner
sys.path.insert(0, str(ROOT / 'Graduation-Project/Ablation_V2'))
from notebook_resource_wait import configure_resource_wait
run_stage = configure_resource_wait(_lock_runner)
ARM = 'no_c8_mmr_demo_selection'
RUN_INFERENCE = True
WORKERS, INFERENCE_WORKERS = 4, 10
config = AblationConfig(root=ROOT, dataset='devign', workers=WORKERS, inference_workers=INFERENCE_WORKERS, run_inference=RUN_INFERENCE)
print('Arm:', ARM, '| enabled:', ARM_SPECS[ARM]['enabled'])
print('Output:', ABLATIONS / ARM_SPECS[ARM]['folder'] / 'test_v2')


Arm: no_c8_mmr_demo_selection | enabled: ['C1', 'C2', 'C3', 'C5', 'C6', 'C7']
Output: /mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Devign/Ablations/No-C8-mmr-demo-selection/test_v2


## Prepare và freeze
Đọc cache nặng tại chỗ; không encode lại và không gọi Joern. Kiểm tra fingerprint trước khi tạo prompt.


In [4]:
prepared = run_stage(config, ARM, 'prepare')
prepared


no_c8_mmr_demo_selection: verifying frozen baseline, canonical cohort and existing caches
no_c8_mmr_demo_selection: prepared 100/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 200/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 300/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 400/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 500/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 600/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 700/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 800/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 900/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 1000/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 1100/2732 using existing CodeT5/Joern caches
no_c8_mmr_demo_selection: prepared 1200

{'schema': 'code_based_ablation_v2',
 'status': 'prepared_before_inference',
 'profile': 'code',
 'arm': 'no_c8_mmr_demo_selection',
 'dataset': 'devign',
 'split': 'test',
 'target_count': 2732,
 'enabled_improvements': ['C1', 'C2', 'C3', 'C5', 'C6', 'C7'],
 'disabled_improvements': ['C8'],
 'C4': 'shared evaluator, always enabled',
 'config': {'variant': 'code',
  'dataset': 'devign',
  'split': 'test',
  'root': '/mnt/c/MY-FILES/LAB-THAY-THO',
  'semantic_top_k': 5,
  'lexical_weight': 0.7,
  'whitening_dim': 256,
  'tsne_dim': 2,
  'tsne_perplexity': 30.0,
  'tsne_iterations': 1000,
  'batch_size': 4,
  'workers': 4,
  'inference_workers': 10,
  'joern_shard_index': 0,
  'joern_shard_count': 1,
  'model_dir': '/mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Reproduced-Code-Based/kaggle_gpu_job/downloads/quangdung1111-grace-kaggle-gpu-ipynb/downloads/model',
  'max_targets': None,
  'reuse_legacy_reveal_test_embeddings': False},
 'source_run_manifest': '/mnt/c/MY-FILES/L

## Inference và evaluation
Cell này gọi Vertex thật nếu RUN_INFERENCE=True. Checkpoint mới nằm trong test_v2.


In [6]:
if RUN_INFERENCE:
    metrics = run_stage(config, ARM, 'inference')
    result = run_stage(config, ARM, 'evaluate')
    print(result['status'], result['evaluation'])
else:
    print('Preparation/freeze complete; RUN_INFERENCE=False')


inference: 2732/2732 cached; 0 Vertex workers
complete {'n_responses': 2732, 'n_valid': 2732, 'n_invalid': 0, 'invalid_rate': 0.0, 'confusion_matrix': {'tp': 364, 'tn': 1136, 'fp': 341, 'fn': 891}, 'accuracy_valid': 0.5490483162518301, 'precision': 0.5163120567375886, 'recall': 0.2900398406374502, 'f1': 0.3714285714285714, 'balanced_accuracy': 0.5295832243133087, 'mcc': 0.06738545729314367, 'predicted_vulnerable_rate_valid': 0.25805270863836016}
